<a href="https://colab.research.google.com/github/jvkvasanth/Spark/blob/main/SparkQueryPlan_Repartition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Wide Transformations

  1. Repartitions
  2. Coalesce
  3. Joins
  4. GroupBy
      *   count
      *   countDistinct
      *   sum









1. **Spark query plan for repartition**






In [4]:
!pip install findspark

In [2]:
!git clone https://github.com/jvkvasanth/Spark.git

Cloning into 'Spark'...
remote: Enumerating objects: 397, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 397 (delta 8), reused 1 (delta 0), pack-reused 378 (from 1)
Receiving objects: 100% (397/397), 702.55 MiB | 42.27 MiB/s, done.
Resolving deltas: 100% (23/23), done.
Updating files: 100% (345/345), done.


In [5]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext

In [ ]:
transaction_df = spark.read.parquet("/content/Spark/data/data_skew/transactions.parquet");
customer_df = spark.read.parquet("/content/Spark/data/data_skew/customers.parquet");

In [ ]:
trans_partition_count = transaction_df.rdd.getNumPartitions()
customer_partition_count = customer_df.rdd.getNumPartitions()
print("Transactions Partition Count: ", trans_partition_count)
print("Customers Partition Count: ", customer_partition_count)

Transactions Partition Count:  12
Customers Partition Count:  1


How spark arrived 12 partitions though we have so many small parquet files?

Spark's default behavior when reading Parquet files is to determine the number of partitions based on the total size of the data and the block size of the underlying file system, rather than the number of individual files.

Here's a more detailed explanation:

InputFormat and Splitting: When Spark reads data from a distributed file system (like HDFS or even a local file system in your case), it uses an InputFormat to define how the data is split into smaller chunks that can be processed in parallel. For Parquet, Spark uses an InputFormat that considers the file structure and metadata.
Block Size and Splits: Parquet files are typically stored in blocks. Spark's InputFormat will create one "split" for each block (or a combination of blocks, depending on configuration). A split is a logical division of the input data that a single task will process. The number of these splits determines the initial number of partitions in your DataFrame.
Default Parallelism: Spark also has a default level of parallelism, which can influence the final number of partitions. This is often related to the number of cores available in your Spark cluster (or on your local machine if you're running in local mode).
Why 12 Partitions? In your case, even though you have many small Parquet files, Spark likely aggregated their sizes and determined that the total data could be efficiently processed using 12 partitions based on the block size or other configuration settings. It doesn't necessarily create one partition per file, especially if the files are very small. Spark tries to create partitions that are a reasonable size for processing (e.g., around 128MB or 256MB by default).
In summary, Spark's partitioning when reading Parquet is driven by the total data size and how it can be split into manageable blocks for parallel processing, not simply the count of individual files.



In [ ]:
transaction_df.repartition(4).explain(True)

== Parsed Logical Plan ==
Repartition 4, true
+- Relation [cust_id#22,start_date#23,end_date#24,txn_id#25,date#26,year#27,month#28,day#29,expense_type#30,amt#31,city#32] parquet

== Analyzed Logical Plan ==
cust_id: string, start_date: string, end_date: string, txn_id: string, date: string, year: string, month: string, day: string, expense_type: string, amt: string, city: string
Repartition 4, true
+- Relation [cust_id#22,start_date#23,end_date#24,txn_id#25,date#26,year#27,month#28,day#29,expense_type#30,amt#31,city#32] parquet

== Optimized Logical Plan ==
Repartition 4, true
+- Relation [cust_id#22,start_date#23,end_date#24,txn_id#25,date#26,year#27,month#28,day#29,expense_type#30,amt#31,city#32] parquet

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=27]
   +- FileScan parquet [cust_id#22,start_date#23,end_date#24,txn_id#25,date#26,year#27,month#28,day#29,expense_type#30,amt#31,city#32] Batched: true, DataF

**Physical plan explained**

The Physical Plan is the detailed, low-level plan that Spark generates to execute your query on the cluster. It describes the actual operations that will be performed on the data, including how data will be moved and processed across the nodes.

Here's an explanation of the key parts in your output:

AdaptiveSparkPlan isFinalPlan=false: This indicates that Adaptive Query Execution (AQE) is enabled. AQE is a feature in Spark that can dynamically adjust the query plan during execution based on runtime statistics. isFinalPlan=false means this is not the final plan yet and might be optimized further by AQE.
+- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=27]: This is the most important part of this plan output and directly relates to the repartition(4) operation you performed.
Exchange: This is a physical operator that represents a data shuffle across the network between different stages of a Spark job. Shuffles are expensive operations as they involve moving data between executors.
RoundRobinPartitioning(4): This specifies the partitioning strategy being used. RoundRobinPartitioning is a method of distributing data evenly across a fixed number of partitions. In this case, the data is being repartitioned into 4 partitions as you specified in your repartition(4) call. Each row is assigned to a partition in a round-robin fashion.
REPARTITION_BY_NUM: This further clarifies that the exchange is happening because of a repartitioning operation based on the number of partitions.
[plan_id=27]: This is an internal identifier for this specific plan node.
+- FileScan parquet [cust_id#22,start_date#23,end_date#24,txn_id#25,date#26,year#27,month#28,day#29,expense_type#30,amt#31,city#32] ...: This part describes the initial operation of reading the data from the Parquet file.
FileScan parquet [...]: Indicates that Spark is performing a file scan to read data from a Parquet source.
[cust_id#22,start_date#23,end_date#24,txn_id#25,date#26,year#27,month#28,day#29,expense_type#30,amt#31,city#32]: These are the columns being read from the Parquet file, along with their internal Spark identifiers.
The rest of the details (Batched, DataFilters, Format, Location, PartitionFilters, PushedFilters, ReadSchema) provide more information about how the file is being read, including whether batch reading is enabled, any filters being applied (none in this case), the file format, the location of the file, partition filters (none in this case), pushed-down filters (none in this case), and the schema being read.
In essence, this physical plan shows that Spark will first read the Parquet file (FileScan) and then perform a shuffle operation (Exchange) using the RoundRobinPartitioning strategy to redistribute the data into 4 partitions as a result of your repartition(4) call. This repartitioning forces a wide transformation, requiring data movement between executors.